# Tome 8, Chapter 1: Smooth Manifolds (Python companion)

The inverse function theorem as an algorithm, inverse kinematics of robot arms and the self-motion of a redundant arm, and the dimensions of matrix groups computed from ranks.

*Before each solution, predict what the output will look like.*

## Exercises

**P1.1 (★) The proof of the inverse function theorem is an algorithm.** The proof solves $f(x)=y$ by iterating $x\mapsto x+A^{-1}(y-f(x))$ with the *fixed* matrix $A=Df(a)$ (the chord method), while Newton's method uses $A=Df(x)$ at the current point. Take $f(x,y)=(e^x\cos y,e^x\sin y)$, $a=(0,0)$ and targets $w$ at distance $0.1$, $0.5$ and $0.9$ from $f(a)=(1,0)$. (a) Run both methods and plot $|x_k-x^*|$ against $k$ on a log scale. (b) Measure the contraction ratio of the chord method and compare it with $\sup\|I-A^{-1}Df\|$ along the path. (c) Explain why the chord method is linearly convergent and Newton quadratically, and why the proof uses the slower one.

**P1.2 (★★) CS application: inverse kinematics of robot arms.** (a) For the two-link arm of Exercise 1.3 with $\ell_1=1$, $\ell_2=0.6$, compute both closed-form solutions for a target $w$ and check them. (b) Implement damped least squares, $\Delta\theta=J^\top(JJ^\top+\lambda^2I)^{-1}(w-E(\theta))$, and run it from many random starting angles: which posture does it find, and how does the answer depend on the start? (c) Move the target towards the outer circle $|w|=1.6$ and watch the condition number of $J$ and the number of iterations. (d) A three-link planar arm $E\colon T^3\to\R^2$ is redundant: by the regular value theorem the set of postures reaching a regular value $w$ is a curve. Trace it by moving along $\ker J$ and correcting back with a Newton step, and plot the self-motion of the arm.

**P1.3 (★) Dimensions of matrix groups, by computer.** For $n=2,\dots,6$: (a) take a random point $X$ of $\mathrm O(n)$ (QR of a Gaussian matrix), write the linear map $H\mapsto H^\top X+X^\top H$ as a matrix and compute its rank and the dimension of its kernel; compare with $\frac{n(n-1)}2$. (b) Do the same for $\mathrm{SL}_n(\R)$, $\mathrm U(n)$ and $\mathrm{SU}(n)$ (Exercise 1.2). (c) At $X=I$, check numerically that the kernel of the derivative is the set of skew-symmetric matrices, and that $\exp(tH)\in\mathrm O(n)$ for skew $H$: a first glimpse of Chapter 7.

## Solutions

### P1.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(p):
    x, y = p
    return np.array([np.exp(x) * np.cos(y), np.exp(x) * np.sin(y)])

def Df(p):
    x, y = p
    e = np.exp(x)
    return np.array([[e * np.cos(y), -e * np.sin(y)], [e * np.sin(y), e * np.cos(y)]])

a = np.zeros(2)
A_inv = np.linalg.inv(Df(a))

def solve(w, method, steps=40):
    p = a.copy(); path = [p.copy()]
    for _ in range(steps):
        M = A_inv if method == "chord" else np.linalg.inv(Df(p))
        p = p + M @ (w - f(p))
        path.append(p.copy())
    return np.array(path)

fig, ax = plt.subplots(figsize=(6.5, 4))
for d, c in zip((0.1, 0.5, 0.9), ("C0", "C1", "C2")):
    w = np.array([1 + d * np.cos(1.0), d * np.sin(1.0)])         # a target at distance d from (1,0)
    z = complex(*w); exact = np.array([np.log(abs(z)), np.angle(z)])  # the true preimage: the logarithm
    for method, ls in (("chord", "-"), ("newton", "--")):
        err = np.linalg.norm(solve(w, method) - exact, axis=1)
        ax.semilogy(np.maximum(err, 1e-17), ls, color=c, label=f"{method}, |w-f(a)|={d}")
        if method == "chord":
            ratios = err[2:12] / err[1:11]
            # ||I - A^{-1} Df|| along the segment from a to the solution
            ts = np.linspace(0, 1, 50)
            sup = max(np.linalg.norm(np.eye(2) - A_inv @ Df(t * exact), 2) for t in ts)
            print(f"d={d}: chord ratio ~ {ratios.mean():.3f}, sup ||I - A^-1 Df|| on the path = {sup:.3f}")
ax.set_xlabel("iteration k"); ax.set_ylabel("error"); ax.legend(fontsize=7); ax.set_ylim(1e-17, 2)
plt.show()

(a) Newton (dashed) reaches machine precision in four to six steps for every target: the number of correct digits roughly doubles at each step. The chord method (solid) is a straight line on the log scale: linear convergence, fast for the target at distance $0.1$ (ratio about $0.1$), slower at $0.5$ (about $0.5$), and painfully slow at $0.9$ (about $0.9$).

(b) The observed ratio is close to $\sup\|I-A^{-1}Df\|$ on the segment from $a$ to the solution (slightly above it for the farthest target, whose iterates overshoot the segment), which is the Lipschitz constant of $T_y$ used in the proof. Here $A^{-1}Df(p)=e^x R_y$ is a scaled rotation, so $\|I-A^{-1}Df\|=|1-e^{x+iy}|$, which grows with the distance from $a$: the guarantee $\le\frac12$ of the proof holds only on a small ball, exactly as the proof assumed.

(c) The chord map has derivative $I-A^{-1}Df(x^*)$ at the solution, which is non-zero, so the error is multiplied by roughly that much at each step. Newton's map has derivative zero at the solution, so the error is squared. The proof uses the chord method because its contraction constant is controlled uniformly by $\|Df-I\|\le\frac12$ on a fixed ball, for all targets at once, which is what produces the open set $W$; Newton's convergence is faster but its basin is harder to describe, and its iteration map depends on $Df$ at moving points.

### P1.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

l1, l2 = 1.0, 0.6

def E(t):
    return np.array([l1 * np.cos(t[0]) + l2 * np.cos(t[0] + t[1]),
                     l1 * np.sin(t[0]) + l2 * np.sin(t[0] + t[1])])

def J(t):
    s1, c1 = np.sin(t[0]), np.cos(t[0]); s12, c12 = np.sin(t[0] + t[1]), np.cos(t[0] + t[1])
    return np.array([[-l1 * s1 - l2 * s12, -l2 * s12], [l1 * c1 + l2 * c12, l2 * c12]])

def closed_form(w):
    r2 = w @ w
    c2 = (r2 - l1**2 - l2**2) / (2 * l1 * l2)
    sols = []
    for sgn in (+1, -1):
        t2 = sgn * np.arccos(np.clip(c2, -1, 1))
        u = complex(l1 + l2 * np.cos(t2), l2 * np.sin(t2))
        t1 = np.angle(complex(*w) / u)
        sols.append(np.array([t1, t2]))
    return sols

w = np.array([0.95, 0.75])
for s in closed_form(w):
    print("closed form:", np.round(s, 4), " E(theta) - w =", np.round(E(s) - w, 12),
          " posture:", "elbow down" if s[1] > 0 else "elbow up")

def dls(w, t0, lam=0.05, tol=1e-12, maxit=500):
    t = t0.copy()
    for k in range(maxit):
        e = w - E(t)
        if np.linalg.norm(e) < tol:
            return t, k
        Jt = J(t)
        t = t + Jt.T @ np.linalg.solve(Jt @ Jt.T + lam**2 * np.eye(2), e)
    return t, maxit

rng = np.random.default_rng(0)
found = {"elbow down": 0, "elbow up": 0}
starts = rng.uniform(-np.pi, np.pi, size=(400, 2))
labels = []
for t0 in starts:
    t, k = dls(w, t0)
    t2 = (t[1] + np.pi) % (2 * np.pi) - np.pi
    lab = "elbow down" if t2 > 0 else "elbow up"
    found[lab] += 1; labels.append(t2 > 0)
print(found)
labels = np.array(labels)
plt.figure(figsize=(4.5, 4.2))
plt.scatter(starts[labels, 0], starts[labels, 1], s=6, label="converges to elbow down")
plt.scatter(starts[~labels, 0], starts[~labels, 1], s=6, label="converges to elbow up")
plt.xlabel(r"$\theta_1$ at start"); plt.ylabel(r"$\theta_2$ at start"); plt.legend(fontsize=7)
plt.title("Basins of the two postures"); plt.show()

In [ ]:
# (c) towards the outer circle |w| = l1 + l2 = 1.6: the two solutions merge and J becomes singular.
direction = w / np.linalg.norm(w)
for r in (1.2, 1.5, 1.58, 1.599, 1.5999, 1.6):
    target = r * direction
    t, k = dls(target, np.array([0.3, 0.8]), lam=0.0 if r < 1.6 else 0.05, maxit=2000)
    cond = np.linalg.cond(J(closed_form(target)[0]))
    print(f"|w| = {r:7.4f}: cond(J) at the solution = {cond:12.1f}   iterations = {k:5d}   "
          f"residual = {np.linalg.norm(E(t) - target):.1e}")

(a) The two closed-form solutions reproduce $w$ to rounding error; with $\theta_2>0$ the elbow bends to the right of the forearm direction (the arm in Figure 1.2 labelled elbow down) and with $\theta_2<0$ to the left.

(b) Damped least squares converges from every start, about half the time to each posture (204 against 196 here), but *which* posture it finds depends on the start in a complicated way: the plot shows two interleaved basins, and the sign of the starting elbow angle predicts the outcome only about $60\%$ of the time, because a first large step can carry the elbow across the critical lines $\theta_2\in\{0,\pi\}$ of Exercise 1.3. A controller that must not flip the elbow mid-motion therefore starts each solve from the previous posture and takes small steps.

(c) As $|w|\to\ell_1+\ell_2$ the condition number of $J$ at the solution blows up, roughly like the inverse square root of the distance to the circle, and is infinite on it. Undamped Newton ($\lambda=0$) still converges inside, but needs more iterations and loses digits in the residual, because the matrix it inverts is nearly singular. Exactly on the circle the undamped iteration would divide by zero; the damping $\lambda$ keeps the steps bounded, but the convergence is no longer quadratic: after $2000$ iterations the residual is still about $2\cdot10^{-6}$. Robotics textbooks call these the *workspace-boundary singularities*.

### P1.2 (d)

In [ ]:
# (d) a redundant planar arm with three links: the postures reaching w form a curve in T^3.
L = np.array([1.0, 0.8, 0.5])

def E3(t):
    c = np.cumsum(t)
    return np.array([np.sum(L * np.cos(c)), np.sum(L * np.sin(c))])

def J3(t):
    c = np.cumsum(t)
    # moving joint j turns every link from j on
    return np.array([[-np.sum(L[j:] * np.sin(c[j:])) for j in range(3)],
                     [ np.sum(L[j:] * np.cos(c[j:])) for j in range(3)]])

def newton_correct(t, w):
    for _ in range(20):
        e = w - E3(t)
        if np.linalg.norm(e) < 1e-13:
            break
        t = t + np.linalg.pinv(J3(t)) @ e
    return t

w3 = np.array([1.2, 0.6])
t = newton_correct(np.array([0.2, 0.9, 0.9]), w3)
path = [t.copy()]
v_prev = None
for step in range(4000):
    _, s, Vt = np.linalg.svd(J3(t))
    v = Vt[-1]                                   # unit vector spanning ker J3(t)
    if v_prev is not None and v @ v_prev < 0:    # keep a consistent orientation
        v = -v
    v_prev = v
    t = newton_correct(t + 0.01 * v, w3)
    path.append(t.copy())
    if step > 50 and np.linalg.norm(((t - path[0] + np.pi) % (2 * np.pi)) - np.pi) < 0.008:
        break
path = np.array(path)
print("points on the self-motion curve:", len(path),
      " max residual:", max(np.linalg.norm(E3(p) - w3) for p in path))
print("smallest singular value of J3 along the curve:", min(np.linalg.svd(J3(p))[1][-1] for p in path))

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for p in path[::len(path) // 14]:
    c = np.cumsum(p)
    pts = np.vstack([[0, 0], np.cumsum(np.c_[L * np.cos(c), L * np.sin(c)], axis=0)])
    ax[0].plot(pts[:, 0], pts[:, 1], "-o", ms=3, alpha=0.6)
ax[0].plot(*w3, "k*", ms=12); ax[0].set_aspect("equal"); ax[0].set_title("the arm, keeping its hand still")
ax[1].plot(np.unwrap(path[:, 0]), np.unwrap(path[:, 2]), ".", ms=2)
ax[1].set_xlabel(r"$\theta_1$"); ax[1].set_ylabel(r"$\theta_3$"); ax[1].set_title("self-motion curve (projection)")
plt.tight_layout(); plt.show()

The tracer walks along $\ker J$, the tangent line of the curve $E_3^{-1}(w)$ given by Theorem 1.30 ($T_pS=\ker dF_p$), and Newton's method projects it back onto the curve after each step. The residual stays at rounding level, the smallest singular value of $J$ never approaches $0$ (so $w$ is a regular value and the theorem applies), and after about $1600$ steps the tracer returns to its start: the self-motion set is a closed curve, a circle embedded in the torus $T^3$. Physically, the hand stays fixed while the elbow and wrist sweep through a one-parameter family of postures. Redundant arms use this freedom to avoid obstacles or joint limits without moving the tool, and the regular value theorem is the reason the freedom is a smooth curve and not something worse.

### P1.3

In [ ]:
import numpy as np
from scipy.linalg import expm

rng = np.random.default_rng(1)

def matrix_of(linear_map, n, field=float):
    """Matrix of a real-linear map on n x n matrices (complex ones seen as R^{2n^2})."""
    cols = []
    basis = []
    for i in range(n):
        for j in range(n):
            E = np.zeros((n, n), dtype=complex if field is complex else float); E[i, j] = 1
            basis.append(E)
            if field is complex:
                E2 = np.zeros((n, n), dtype=complex); E2[i, j] = 1j
                basis.append(E2)
    for B in basis:
        out = linear_map(B)
        cols.append(np.concatenate([out.real.ravel(), out.imag.ravel()]) if field is complex else out.ravel())
    return np.array(cols).T

def rank(M):
    s = np.linalg.svd(M, compute_uv=False)
    return int((s > 1e-9 * s[0]).sum())

print(" n | O(n) kernel  n(n-1)/2 | SL_n kernel  n^2-1 | U(n) kernel  n^2 | SU(n) kernel  n^2-1")
for n in range(2, 7):
    X, _ = np.linalg.qr(rng.standard_normal((n, n)))                       # a point of O(n)
    dO = matrix_of(lambda H: H.T @ X + X.T @ H, n)
    Y = rng.standard_normal((n, n)); Y /= np.sign(np.linalg.det(Y)) * abs(np.linalg.det(Y)) ** (1 / n)  # det Y = 1
    Yinv = np.linalg.inv(Y)
    dSL = matrix_of(lambda H: np.array([[np.trace(Yinv @ H)]]), n)        # D det(Y)H = det Y tr(Y^-1 H)
    Z, _ = np.linalg.qr(rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n)))  # a point of U(n)
    dU = matrix_of(lambda H: H.conj().T @ Z + Z.conj().T @ H, n, complex)
    Zinv = Z.conj().T
    # SU(n): the equations X*X = I and the argument of det, whose derivative at Z (in U(n)) is Im tr(Z^-1 H)
    dSU = matrix_of(lambda H: np.concatenate([(H.conj().T @ Z + Z.conj().T @ H).ravel(),
                                              [1j * np.trace(Zinv @ H).imag]]), n, complex)
    k = lambda M: M.shape[1] - rank(M)
    print(f" {n} |   {k(dO):3d}        {n*(n-1)//2:3d}   |   {k(dSL):3d}       {n*n-1:3d}  |   {k(dU):3d}      {n*n:3d} |"
          f"   {k(dSU):3d}        {n*n-1:3d}")

In [ ]:
# (c) at X = I the kernel of H -> H^T + H is the skew-symmetric matrices, and exp maps them into O(n)
n = 4
dO_I = matrix_of(lambda H: H.T + H, n)
_, s, Vt = np.linalg.svd(dO_I)
kernel = Vt[rank(dO_I):]
print("kernel dimension:", len(kernel), "  every kernel vector skew:",
      all(np.allclose(v.reshape(n, n), -v.reshape(n, n).T) for v in kernel))
H = rng.standard_normal((n, n)); H = H - H.T
for t in (0.1, 1.0, 10.0):
    Q = expm(t * H)
    print(f"t={t:5.1f}: ||Q^T Q - I|| = {np.linalg.norm(Q.T @ Q - np.eye(n)):.1e},  det Q = {np.linalg.det(Q):.6f}")

(a)--(b) In every case the kernel of the derivative has exactly the predicted dimension: $\frac{n(n-1)}2$ for $\mathrm O(n)$, $n^2-1$ for $\mathrm{SL}_n(\R)$, $n^2$ for $\mathrm U(n)$ and $n^2-1$ for $\mathrm{SU}(n)$. Equivalently, the derivative of the defining equations has full rank, which is the hypothesis of the regular value theorem, and by Proposition 1.22 the kernel is the tangent space. For $\mathrm{SL}_n$ we used Jacobi's formula $D\det(Y)H=\det Y\operatorname{tr}(Y^{-1}H)$, of which the exercise only needed the special case $H=Y$; for $\mathrm{SU}(n)$ the extra equation is the derivative of the argument of the determinant.

(c) At the identity the kernel consists of the $6$ skew-symmetric $4\times4$ matrices, and $\exp(tH)$ stays exactly orthogonal, with determinant $1$, for all $t$: the exponential maps the tangent space at $I$ into the group, and lands in $\mathrm{SO}(n)$. Chapter 7 makes this the definition of the Lie algebra and proves that the exponential is a local diffeomorphism at $0$, using the inverse function theorem of this chapter.